# Dynamic-Mode Visualization

Thin plotting workflow for standardized ChronoPack gene-trajectory clustering outputs. Reusable plotting logic lives in `chronopack.pl`; this notebook keeps only dataset-specific selections and output paths.

In [1]:
import os
from pathlib import Path

import anndata as ad
import chronopack as cp

# Folder holding one sub-folder per experiment; see analyses/README.md.
DATA_DIR = Path(os.environ.get("CHRONOPACK_DATA_DIR", "../data"))
OUTPUT_DIR = DATA_DIR / "gene_clustering"
PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# One row per gene and context; fitted curves in X, cluster labels in obs.
adata = ad.read_h5ad(OUTPUT_DIR / "gene_trajectory_clusters.h5ad")

print(adata)
print(adata.obs["broad_cluster"].value_counts())

AnnData object with n_obs × n_vars = 5091 × 97
    obs: 'gene', 'cell_type', 'waldStat_1', 'meanLogFC', 'pvalue_1', 'qval_1', 't_onset', 't_peak', 'baseline_exp', 'peak_exp', 'peak_dlogexp', 'peak_dexp', 'peak_log2fc', 'auc_norm', 'owp', 'owp_ddt', 'mean_curv', 'max_curv', 'mean_turn', 'max_turn', 'cluster_broad_induced', 'cluster_dist_broad_induced', 'cluster_fine_induced', 'cluster_dist_fine_induced', 'cluster_broad_repressed', 'cluster_dist_broad_repressed', 'cluster_fine_repressed', 'cluster_dist_fine_repressed', 'broad_cluster', 'cluster_dist_broad', 'cluster_dist_fine'
    var: 'time'
    uns: 'chronopack'
broad_cluster
trend-down_1      1521
trend-up          1478
trend-down_2       897
impulse-up_2       474
impulse-up_1       472
transient-down     249
Name: count, dtype: int64


## Broad Dynamic Modes

In [2]:
broad_order = [
    "trend-up",
    "impulse-up_1",
    "impulse-up_2",
    "trend-down_1",
    "trend-down_2",
    "transient-down",
]

broad = cp.pl.cluster_trajectories(
    adata,
    groupby="broad_cluster",
    normalize=True,
    output=PLOT_DIR / "broad_dynamic_modes.svg",
    title="Broad dynamic modes: median +/- IQR",
)
broad.output

'/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/broad_dynamic_modes.svg'

## Representative Genes

In [3]:
def top_rows_by_broad_cluster(adata, broad_clusters, n=3):
    rows = []
    for mode in broad_clusters:
        sub = adata.obs.loc[adata.obs["broad_cluster"].astype(str) == mode, ["waldStat_1"]].copy()
        sub["row_id"] = sub.index.astype(str)
        rows.extend(sub.sort_values(["waldStat_1", "row_id"], ascending=[False, True]).head(n).index.tolist())
    return rows

broad_examples = top_rows_by_broad_cluster(adata, broad_order, n=3)
examples = cp.pl.gene_trajectory(
    adata,
    broad_examples,
    normalize=False,
    output=PLOT_DIR / "broad_cluster_example_genes.svg",
    width=12,
    height=12,
    title="Top example genes from each broad cluster",
)
examples.output

'/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/broad_cluster_example_genes.svg'

## Fine Clusters Within Broad Modes

In [4]:
def fine_column_for_broad(mode):
    if mode in {"trend-up", "impulse-up_1", "impulse-up_2"}:
        return "cluster_fine_induced"
    return "cluster_fine_repressed"

fine_outputs = {}
for mode in broad_order:
    fine_col = fine_column_for_broad(mode)
    rows = adata.obs_names[
        (adata.obs["broad_cluster"].astype(str) == mode)
        & (adata.obs[fine_col].astype(str) != "-1")
    ].tolist()
    if rows:
        res = cp.pl.cluster_trajectories(
            adata,
            rows=rows,
            groupby=fine_col,
            normalize=True,
            output=PLOT_DIR / f"fine_clusters.{mode}.svg",
            title=f"{mode}: fine-cluster median curves",
        )
        fine_outputs[mode] = res.output
fine_outputs

{'trend-up': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/fine_clusters.trend-up.svg', 'impulse-up_1': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/fine_clusters.impulse-up_1.svg', 'impulse-up_2': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/fine_clusters.impulse-up_2.svg', 'trend-down_1': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/fine_clusters.trend-down_1.svg', 'trend-down_2': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/fine_clusters.trend-down_2.svg', 'transient-down': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/fine_clusters.transient-down.svg'}

## Trajectory Feature Space

In [5]:
features = cp.pl.trajectory_features(
    adata,
    x="t_peak",
    y="peak_log2fc",
    color="broad_cluster",
    output=PLOT_DIR / "trajectory_feature_space.svg",
    title="Peak time vs. peak log2 fold change",
)
features.output

'/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/trajectory_feature_space.svg'

## Fine-Cluster Representative Table

In [6]:
fine_examples = []
for mode in broad_order:
    fine_col = fine_column_for_broad(mode)
    rows_bc = adata.obs_names[adata.obs["broad_cluster"].astype(str) == mode]
    for fine in sorted(set(adata.obs.loc[rows_bc, fine_col].astype(str)) - {"-1"}):
        sub = adata.obs.loc[
            rows_bc[adata.obs.loc[rows_bc, fine_col].astype(str) == fine],
            ["waldStat_1"],
        ].copy()
        sub["row_id"] = sub.index.astype(str)
        fine_examples.extend(sub.sort_values(["waldStat_1", "row_id"], ascending=[False, True]).head(1).index.tolist())

fine_example_plot = cp.pl.gene_trajectory(
    adata,
    fine_examples,
    normalize=False,
    output=PLOT_DIR / "fine_cluster_example_genes.svg",
    width=14,
    height=14,
    title="Top representative gene from each fine cluster",
)
fine_example_plot.output

'/new-stg/home/alex/tmp/ChronoPack/verification_runs/full_pipeline_analyses_20260930_r9/data/gene_clustering/plots/fine_cluster_example_genes.svg'